In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import importlib
import dataset
from image_builder import *
from data_prepare import *
import data_fetch
from data_fetch import *
from data_fetch import _download_single, _read_mops_csv

importlib.reload(dataset)
importlib.reload(data_fetch)

<module 'data_fetch' from '/home/iof_314707035/Jasper/Reimage_price_trends/data_fetch.py'>

In [2]:
from dataset import *
import os

START = "1993-01-01"
END = "2019-12-31"
TRAIN_END = "2000-12-31"
SAVE_DIR = "data"
Market = "US_1993_2020"
I = 5
R = 5
CRSP_DATA_DIR = resolve_crsp_data_dir()
print("CRSP data dir:", CRSP_DATA_DIR)

TICKERS, crsp_info = get_crsp_permno_universe(
    crsp_data_dir=CRSP_DATA_DIR,
    start=START,
    end=END,
    return_info=True,
)

print("permno 數量:", len(TICKERS))
display(crsp_info.head())

CRSP data dir: us_stock
permno 數量: 20721


,ticker,permno,first_year,last_year,rows
0,10001,10001,1993,2017,5666
1,10002,10002,1993,2013,3873
2,10003,10003,1993,1995,665
3,10009,10009,1993,2000,1347
4,10010,10010,1993,1995,670


In [3]:

# X_trainval, y_trainval, X_test, y_test, meta = build_research_dataset(
#     tickers=TICKERS,
#     start=START,
#     end=END,
#     I=I,
#     R=R,
#     train_end=TRAIN_END,
#     save_dir=SAVE_DIR,
#     Market=Market,
#     price_source="crsp",
#     crsp_data_dir=CRSP_DATA_DIR,
#     checkpoint_every=1,
#     resume=False,
#     process_by="year",
#     checkpoint_save_arrays=False,
#     year_ticker_chunk_size=1000,
#     max_workers=8,
# )



In [4]:
I = 5
R = 5
Market = "US_1993_2020"

dataset_dir = f"data/training_data/{Market}/I{I}R{R}S{R}_week"

X_trainval = np.load(f"{dataset_dir}/X_trainval.npy")
y_trainval = np.load(f"{dataset_dir}/y_trainval.npy")

X_test = np.load(f"{dataset_dir}/X_test.npy")
y_test = np.load(f"{dataset_dir}/y_test.npy")

meta = pd.read_csv(f"{dataset_dir}/meta.csv")
meta_trainval = pd.read_csv(f"{dataset_dir}/meta_trainval.csv")
meta_test = pd.read_csv(f"{dataset_dir}/meta_test.csv")

print("X_trainval:", X_trainval.shape)
print("y_trainval:", y_trainval.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("meta_trainval:", meta_trainval.shape)
print("meta_test:", meta_test.shape)

display(meta.head())



X_trainval: (3177623, 32, 15)
y_trainval: (3177623,)
X_test: (5899718, 32, 15)
y_test: (5899718,)
meta_trainval: (3177623, 12)
meta_test: (5899718, 12)


,ticker,start_date,date,label_end_date,label,ret,I,R,sample_step,sample_freq,price_source,ending_date
0,10001,1993-01-14,1993-01-22,1993-02-12,0,-0.035703,5,5,5,week,crsp,1993-01-22
1,10001,1993-02-02,1993-02-12,1993-02-26,1,0.036033,5,5,5,week,crsp,1993-02-12
2,10001,1993-02-12,1993-02-26,1993-03-12,1,0.038308,5,5,5,week,crsp,1993-02-26
3,10001,1993-03-08,1993-03-12,1993-04-08,0,-0.070042,5,5,5,week,crsp,1993-03-12
4,10001,1993-03-25,1993-04-08,1993-04-16,1,0.044248,5,5,5,week,crsp,1993-04-08


In [5]:
import model
importlib.reload(model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


RUN_SEEDS = list(range(1))
ENSEMBLE_DIR = f"models/{Market}/I{I}R{R}"
EPOCHS = 50
BATCH_SIZE = 256
VAL_RATIO = 0.3

ensemble_results = model.train_resplit_ensemble(
    X_trainval=X_trainval,
    y_trainval=y_trainval,
    I=I,
    R=R,
    seeds=RUN_SEEDS,
    save_dir=ENSEMBLE_DIR,
    Market=Market,
    val_ratio=VAL_RATIO,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=1e-4,
    weight_decay=0,
    device=device,
)


Using device: cuda


ensemble runs:   0%|                                                              | 0/1 [00:00<?, ?it/s]

ensemble runs:   0%|                                                      | 0/1 [00:00<?, ?it/s, seed=0]

/home/iof_314707035/miniconda3/envs/reimage-price-trends/lib/python3.11/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.5) - (12.0)
    
  queued_call()


seed 0:   0%|                                                                    | 0/50 [00:00<?, ?it/s]

...

seed 0:  52%|▌| 26/50 [46:21<42:51, 107.16s/it, best=55.29%, train_loss=0.6826, val_acc=55.26%, val_loss

seed 0:  52%|▌| 26/50 [48:08<42:51, 107.16s/it, best=55.29%, train_loss=0.6826, val_acc=55.25%, val_loss

seed 0:  52%|▌| 26/50 [48:08<42:51, 107.16s/it, early_stop=epoch 27, train_loss=0.6826, val_acc=55.25%, 

ensemble runs: 100%|████████████████████████████████████████████| 1/1 [48:11<00:00, 2891.81s/it, seed=0]

ensemble runs: 100%|████████████████████████████████████████████| 1/1 [48:11<00:00, 2891.81s/it, seed=0]

In [6]:
# import torch
# import pandas as pd
# import matplotlib.pyplot as plt


# result = pd.read_csv("/home/iof_314707035/Jasper/Reimage_price_trends/models/US_1993_2020/I5R5/US_1993_2020_training_history_I5R5_seed0.csv")

# plt.figure(figsize=(8, 5))
# plt.plot(result["epoch"], result["val_loss"], label="val_loss", marker="o")
# plt.xlabel("Epoch")
# plt.xticks(result["epoch"])
# plt.ylabel("Loss")
# plt.title("Training / Validation Loss")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()


In [7]:
# plt.figure(figsize=(8, 5))
# plt.plot(result["epoch"], result["val_acc"], label="val_acc", marker="o")
# plt.xlabel("Epoch")
# plt.ylabel("Accuracy")
# plt.title("Validation Accuracy")
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.show()

# Test

In [8]:
# from pathlib import Path
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mean_squared_error, accuracy_score, log_loss
# import model
# import importlib
# importlib.reload(model)

# root = Path("/home/iof_314707035/Jasper")

# # 1. 讀取 test data
# market = "US_1993_2020"
# dataset_dir = root / "Reimage_price_trends" / "data" / "training_data" / market / "I5R5S5_week"

# X_test = np.load(dataset_dir / "X_test.npy")
# y_test = np.load(dataset_dir / "y_test.npy")

# print(X_test.shape, y_test.shape)


In [9]:
# model_dir = root / "Reimage_price_trends" / "models" / "US_1993_2020" / "I5R5"
# RUN_SEEDS = list(range(1))
# seeds = list(RUN_SEEDS)

# summary_frames = []
# checkpoint_paths = []
# for seed in seeds:
#     summary_path = model_dir / f"US_1993_2020_training_summary_I5R5_seed{seed}_only_0_1.csv"
#     summary_df = pd.read_csv(summary_path)
#     best_path = Path(summary_df.loc[0, "best_path"])
#     if not best_path.is_absolute():
#         best_path = root / "Reimage_price_trends" / best_path
#     checkpoint_paths.append(str(best_path))
#     summary_frames.append(summary_df.assign(seed=seed, resolved_best_path=str(best_path)))

# ensemble_summary_df = pd.concat(summary_frames, ignore_index=True)
# print("ensemble checkpoints:")
# for path in checkpoint_paths:
#     print(path)


In [10]:
# pred_prob = model.average_ensemble_predictions(
#     X_test,
#     checkpoint_paths,
#     batch_size=256,
#     device="cuda"
# )

# pred_prob = np.asarray(pred_prob).reshape(-1)
# pred_label = (pred_prob > 0.5).astype(int)


In [11]:
# test_mse = mean_squared_error(y_test, pred_prob)
# test_acc = accuracy_score(y_test, pred_label)
# test_logloss = log_loss(y_test, pred_prob)

# print("ensemble checkpoints:", len(checkpoint_paths))
# print("test MSE:", test_mse)
# print("test accuracy:", test_acc)
# print("test logloss:", test_logloss)


In [12]:
# import pandas as pd
# import portfolio_backtest as pb

# pred_df = pb.make_prediction_frame(
#     meta=meta_test,
#     y_true=y_test,
#     pred_prob=pred_prob,
#     pred_label_threshold=0.5,
# )


# seed_tag = "seed" + "_".join(str(seed) for seed in seeds)
# run_dir = model_dir / f"ensemble_{seed_tag}"
# grouped_df, portfolio_returns, summary = pb.run_decile_backtest(
#     pred_df,
#     n_groups=10,
#     R=5,
#     output_dir=str(run_dir / "backtest_ensemble_mean_prob"),
#     prefix="i5r5_ensemble_mean_prob",
# )

# summary

In [13]:
# import re
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# js_df = pd.read_csv("/home/iof_314707035/Jasper/WORK_SPACE/new_model_res/portfolio/USA_D5L2F53S11D11MP21F53S11D11MP21C64_5d5p-lr1E-04-dp0.50-maTrue-vbTrue-weeklyTrained-noDelayedReturn_ensem10_noDelayedReturn/ew.csv")

# fig, ax = plt.subplots(1, 2, figsize=(10, 5.5))
# width = 1 / (len(portfolio_returns.columns) + 1)
# x = np.arange(len(portfolio_returns))

# ax[0].plot(x - width / 2, portfolio_returns["annualized_return"], width, c = 'r', label = 'reimage return')
# ax[0].plot(np.linspace(1,len(js_df['ret'][0:9]), len(js_df['ret'][0:9])), js_df["ret"][0:9], label="js return")
# ax[0].axhline(0, color="black", linewidth=0.8)
# ax[0].set_xticks(x)
# ax[0].set_xticklabels([f"G{g}" for g in portfolio_returns["group"]])
# ax[0].set_xlabel("Portfolio group")
# ax[0].set_ylabel("Annualized value")
# ax[0].set_title("Decile portfolio annualized return")
# ax[0].legend()
# ax[0].grid( alpha=0.25)

# ax[1].plot(x - width / 2, portfolio_returns["annualized_std"], width, c = 'r', label = 'reimage return')
# ax[1].plot(np.linspace(1,len(js_df['std'][0:9]), len(js_df['std'][0:9])), js_df["std"][0:9], label="js return")
# ax[1].set_xticks(x)
# ax[1].set_xticklabels([f"G{g}" for g in portfolio_returns["group"]])
# ax[1].set_xlabel("Portfolio group")
# ax[1].set_ylabel("Annualized value")
# ax[1].set_title("Decile portfolio Standard Deviation")
# ax[1].legend()
# ax[1].grid(alpha=0.25)
# plt.tight_layout()
# plt.show()


In [14]:
# from pathlib import Path
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt


# def find_jasper_root(start):
#     start = Path(start).resolve()
#     for candidate in [start] + list(start.parents):
#         if (candidate / "CACHE_DIR" / "spy_week_ret.csv").exists() and (candidate / "Reimage_price_trends").exists():
#             return candidate
#     raise FileNotFoundError(
#         "Could not find Jasper root containing CACHE_DIR/spy_week_ret.csv and Reimage_price_trends. "
#         f"Current working directory is {start}"
#     )


# jasper_root = find_jasper_root(Path.cwd())
# print("Detected Jasper root:", jasper_root)

# # Reimage portfolio output. Change this path if you use another ensemble output folder.
# reimage_path = jasper_root / "Reimage_price_trends" / "models" / "US_1993_2020" / "I5R5" / "ensemble_seed0" / "backtest_ensemble_mean_prob" / "i5r5_ensemble_mean_prob_portfolio_returns.csv"

# # JS CNN Figure 5 output from portfolio_performance_helper(5, 5). If it has not been generated yet,
# # this cell falls back to the existing JS CNN portfolio file used earlier.
# js_figure5_path = jasper_root / "CACHE_DIR" / "PORTFOLIO" / "cnn_weekly" / "CNN5D5P" / "pf_data" / "pf_data_ew.csv"
# js_fallback_path = jasper_root / "WORK_SPACE" / "new_model_res" / "portfolio" / "USA_D5L2F53S11D11MP21F53S11D11MP21C64_5d5p-lr1E-04-dp0.50-maTrue-vbTrue-weeklyTrained-noDelayedReturn_ensem10_noDelayedReturn" / "pf_data" / "pf_data_ew.csv"
# js_path = js_figure5_path if js_figure5_path.exists() else js_fallback_path
# spy_path = jasper_root / "CACHE_DIR" / "spy_week_ret.csv"

# print("Reimage path:", reimage_path)
# print("JS path:", js_path)
# print("SP500 path:", spy_path)

# missing_paths = [p for p in [reimage_path, js_path, spy_path] if not p.exists()]
# if missing_paths:
#     raise FileNotFoundError("Missing input files:\n" + "\n".join(str(p) for p in missing_paths))

# reimage_raw = pd.read_csv(reimage_path)
# js_raw = pd.read_csv(js_path)
# spy_raw = pd.read_csv(spy_path)
# print("raw shapes:", {"reimage": reimage_raw.shape, "js": js_raw.shape, "spy": spy_raw.shape})
# print("JS columns:", js_raw.columns.tolist())
# print("SP500 columns:", spy_raw.columns.tolist())

# reimage = reimage_raw.copy()
# reimage["date"] = pd.to_datetime(reimage["date"])
# reimage = reimage[["date", "long_top_ret", "group_1_ret", "long_short_ret"]].rename(
#     columns={
#         "long_top_ret": "Reimage_H",
#         "group_1_ret": "Reimage_L",
#         "long_short_ret": "Reimage_H-L",
#     }
# )

# js = js_raw.copy()
# first_col = js.columns[0]
# if first_col != "date":
#     js = js.rename(columns={first_col: "date"})
# js["date"] = pd.to_datetime(js["date"])
# js = js[["date", "9", "0", "H-L"]].rename(
#     columns={"9": "JS_H", "0": "JS_L", "H-L": "JS_H-L"}
# )

# spy = spy_raw.copy()
# spy["date"] = pd.to_datetime(spy["date"])
# spy = spy[["date", "nxt_freq_ewret"]].rename(columns={"nxt_freq_ewret": "SP500"})

# print("date ranges:")
# print("  Reimage:", reimage["date"].min().date(), "to", reimage["date"].max().date(), "rows", len(reimage))
# print("  JS CNN :", js["date"].min().date(), "to", js["date"].max().date(), "rows", len(js))
# print("  SP500  :", spy["date"].min().date(), "to", spy["date"].max().date(), "rows", len(spy))

# perf = reimage.merge(js, on="date", how="inner")
# print("after Reimage x JS merge:", perf.shape)
# perf = perf.merge(spy, on="date", how="inner")
# print("after adding SP500:", perf.shape)
# perf = perf[(perf["date"].dt.year >= 2001) & (perf["date"].dt.year <= 2019)].copy()
# perf = perf.sort_values("date").reset_index(drop=True)
# print("after 2001-2019 filter:", perf.shape)

# if perf.empty:
#     raise ValueError("No overlapping dates among Reimage, JS CNN, and S&P500 returns after filtering.")

# return_cols = ["Reimage_H", "Reimage_L", "Reimage_H-L", "JS_H", "JS_L", "JS_H-L", "SP500"]
# log_perf = perf[["date"]].copy()
# for col in return_cols:
#     log_perf[col] = np.log1p(perf[col].astype(float)).cumsum()

# # Match JS CNN make_portfolio_plot: add previous year-end as the zero starting point.
# prev_year = log_perf["date"].iloc[0].year - 1
# prev_day = pd.to_datetime(f"{prev_year}-12-31")
# start_row = pd.DataFrame([{ "date": prev_day, **{col: 0.0 for col in return_cols} }])
# log_perf = pd.concat([start_row, log_perf], ignore_index=True).sort_values("date").reset_index(drop=True)
# print("log_perf shape:", log_perf.shape)
# display(log_perf.head())

# summary = pd.DataFrame({
#     "annual_return": {col: perf[col].mean() * 52 for col in return_cols},
#     "annual_std": {col: perf[col].std() * np.sqrt(52) for col in return_cols},
#     "sharpe_no_rf": {col: (perf[col].mean() * 52) / (perf[col].std() * np.sqrt(52)) for col in return_cols},
#     "final_cum_log_return": {col: log_perf[col].iloc[-1] for col in return_cols},
#     "final_cum_simple_return": {col: np.expm1(log_perf[col].iloc[-1]) for col in return_cols},
# })

# fig, ax = plt.subplots(figsize=(13, 7))
# ax.plot(log_perf["date"], log_perf["Reimage_H-L"], label="Reimage H-L", linewidth=1.5)
# ax.plot(log_perf["date"], log_perf["JS_H-L"], label="JS CNN H-L", linewidth=1.5)
# ax.plot(log_perf["date"], log_perf["SP500"], label="S&P500", linewidth=1.8, color="goldenrod")
# ax.axhline(0.0, color="black", linewidth=0.8)
# ax.set_title("Cumulative Log Return, JS CNN Method")
# ax.set_xlabel("Date")
# ax.set_ylabel("Cumulative log return")
# ax.grid( alpha=0.25)

# ax.legend()
# plt.tight_layout()
# plt.show()

# print(f"Date range: {perf['date'].min().date()} to {perf['date'].max().date()}")
# print(f"Overlapping periods: {len(perf):,}")
# display(summary)
# log_perf.tail()
